[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/18_feature_selection/03_embedded_methods_and_pitfalls.ipynb)

# Bài 03 — Embedded methods và các bẫy khi chọn feature

**🎯 Mục tiêu**
- Embedded: chọn feature **trong lúc** train — L1 (Lasso, L1-logistic), importance của cây (`SelectFromModel`).
- **Lasso path**: feature "vào" model theo thứ tự nào khi giảm regularization.
- Permutation importance và **shadow features** (ý tưởng Boruta) để tách tín hiệu khỏi nhiễu.
- Bẫy: chọn ngoài CV, feature tương quan, chọn ≠ nhân quả, khi nào **không** nên chọn.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/18_feature_selection

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.datasets import make_friedman1
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.inspection import permutation_importance
from sklearn.linear_model import ElasticNetCV, Lasso, LassoCV, lasso_path
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from ml_course.visualization import set_style

set_style()
rng = np.random.default_rng(0)

## 1. Lasso: L1 đẩy hệ số về đúng 0

Dữ liệu tuyến tính: 8 feature có ích (hệ số giảm dần), 30 feature nhiễu, 300 mẫu.

In [ ]:
n, d = 300, 38
X = rng.normal(size=(n, d))
true_coef = np.r_[np.array([5, 4, 3, 2, 1.5, 1, 0.7, 0.4]), np.zeros(30)]
y = X @ true_coef + rng.normal(0, 2, n)
names = [f"signal{i}" for i in range(8)] + [f"noise{i}" for i in range(30)]
Xs = StandardScaler().fit_transform(X)

alphas, coefs, _ = lasso_path(Xs, y, n_alphas=100)
plt.figure(figsize=(10, 4.5))
for j in range(d):
    plt.plot(np.log10(alphas), coefs[j], color="tab:red" if j < 8 else "lightgray", lw=1.5 if j < 8 else 0.8)
lasso_cv = LassoCV(cv=5, random_state=0).fit(Xs, y)
plt.axvline(np.log10(lasso_cv.alpha_), color="k", ls="--", label=f"alpha chọn bằng CV = {lasso_cv.alpha_:.3f}")
plt.gca().invert_xaxis()
plt.xlabel("log10(alpha) — giảm dần →")
plt.ylabel("hệ số")
plt.title("Lasso path: feature có ích (đỏ) 'vào' model trước, nhiễu (xám) vào sau")
plt.legend()
plt.show()
kept = np.array(names)[lasso_cv.coef_ != 0]
print(f"LassoCV giữ {len(kept)} feature: {sum(k.startswith('signal') for k in kept)} có ích + {sum(k.startswith('noise') for k in kept)} nhiễu")

LassoCV chọn alpha tối ưu cho **dự đoán**, và thường giữ thêm **khá nhiều** feature nhiễu với hệ số nhỏ (ở đây 10) (alpha nhỏ một chút giúp giảm bias của các hệ số lớn). Muốn tập **gọn** hơn: dùng alpha lớn hơn (vd one-standard-error rule) hoặc **stability selection** (Lasso trên nhiều mẫu bootstrap, giữ feature được chọn thường xuyên).

In [ ]:
freq = np.zeros(d)
alpha_sel = lasso_cv.alpha_ * 2
for b in range(50):
    idx = rng.choice(n, n // 2, replace=False)
    freq += Lasso(alpha=alpha_sel).fit(Xs[idx], y[idx]).coef_ != 0
freq /= 50
stable = np.array(names)[freq >= 0.8]
print(f"stability selection (≥ 80% số lần): {list(stable)}")

## 2. `SelectFromModel` với importance của cây

Cây/rừng không cho hệ số = 0 nhưng có importance → giữ feature có importance > ngưỡng (vd trung vị, `"mean"`, hoặc giá trị cụ thể). Bắt được quan hệ phi tuyến — nhưng **ngưỡng là tuỳ tiện**:

In [ ]:
Xf, yf = make_friedman1(n_samples=800, n_features=25, noise=1.0, random_state=2)
fnames = [f"x{i + 1}" for i in range(5)] + [f"noise{i}" for i in range(20)]
sfm = SelectFromModel(RandomForestRegressor(n_estimators=200, random_state=0), threshold="mean").fit(Xf, yf)
print("SelectFromModel (RF, ngưỡng = mean importance):", [f for f, k in zip(fnames, sfm.get_support()) if k])

Ngưỡng "mean" **bỏ sót $x_3$** — một tín hiệu thật nhưng importance thấp hơn trung bình (vì 20 feature nhiễu kéo trung bình xuống, còn vài feature mạnh kéo lên). Cần một ngưỡng khách quan hơn →

## 3. Shadow features: importance có vượt ngưỡng nhiễu không?

Một feature có importance 0.02 — là tín hiệu hay nhiễu? Ý tưởng **Boruta**: thêm **bản sao xáo trộn** (shadow) của mọi feature — chúng chắc chắn vô ích. Giữ feature có importance **cao hơn shadow tốt nhất** một cách nhất quán.

In [ ]:
def shadow_selection(X, y, names, n_rounds=10, seed=0):
    r = np.random.default_rng(seed)
    hits = np.zeros(X.shape[1])
    for k in range(n_rounds):
        shadow = np.apply_along_axis(r.permutation, 0, X)                       # xáo trộn từng cột → phá quan hệ với y
        X_aug = np.hstack([X, shadow])
        X_tr, X_va, y_tr, y_va = train_test_split(X_aug, y, test_size=0.3, random_state=k)
        model = RandomForestRegressor(n_estimators=150, random_state=k).fit(X_tr, y_tr)
        imp = permutation_importance(model, X_va, y_va, n_repeats=3, random_state=k).importances_mean
        hits += imp[: X.shape[1]] > imp[X.shape[1]:].max()
    return pd.Series(hits / n_rounds, index=names)


hit_rate = shadow_selection(Xf, yf, fnames)
hit_rate.sort_values(ascending=False).head(8).plot.barh()
plt.xlabel("tỉ lệ số vòng vượt shadow tốt nhất")
plt.title("Shadow features (ý tưởng Boruta)")
plt.show()
print("giữ (≥ 80%):", list(hit_rate[hit_rate >= 0.8].index))

## 4. Bẫy thường gặp

**(a) Chọn feature ngoài CV** → rò rỉ, đánh giá ảo (chương 15).

**(b) Feature tương quan:** L1 có xu hướng chọn **một** trong nhóm tương quan (tuỳ tiện), bỏ các feature khác — dù chúng mang cùng thông tin. Elastic Net (L1 + L2) chọn cả nhóm ổn định hơn.

In [ ]:
z = rng.normal(size=(400, 1))
Xc = np.hstack([z + rng.normal(0, 0.1, (400, 3)), rng.normal(size=(400, 5))])        # 3 bản đo của cùng một đại lượng + 5 nhiễu
yc = 3 * z.ravel() + rng.normal(0, 1, 400)
lasso = LassoCV(cv=5, random_state=0).fit(Xc, yc)
enet = ElasticNetCV(l1_ratio=0.3, cv=5, random_state=0).fit(Xc, yc)
print("Lasso hệ số 3 bản đo:      ", lasso.coef_[:3].round(3))
print("ElasticNet hệ số 3 bản đo: ", enet.coef_[:3].round(3))

**(c) "Không được chọn" ≠ "không liên quan", "được chọn" ≠ "nguyên nhân".** Feature bị bỏ có thể chỉ vì một feature tương quan đã thay thế nó; feature được chọn có thể chỉ là **proxy** của nguyên nhân thật (số lượng kem bán ra "dự đoán" đuối nước — qua nhiệt độ). Diễn giải nhân quả cần thiết kế khác (chương 38).

**(d) Có khi không cần chọn — và khi nào chọn giúp nhiều nhất:** Ridge/Elastic Net và gradient boosting xử lý nhiều feature tốt; chọn feature có thể **làm tệ đi** nếu bỏ nhầm tín hiệu yếu nhưng có ích. Chọn khi: dữ liệu ít so với số feature, chi phí thu thập feature cao, cần model nhỏ/dễ giải thích, feature nhiễu làm hỏng model nhạy cảm (KNN, SVM RBF).

In [ ]:
# (d) với Friedman: RF dùng mọi feature vs RF trên feature đã chọn bằng shadow
keep_idx = [fnames.index(f) for f in hit_rate[hit_rate >= 0.8].index]
cv = KFold(5, shuffle=True, random_state=0)
for name, Xuse in [("RF, 25 feature", Xf), (f"RF, {len(keep_idx)} feature (shadow)", Xf[:, keep_idx])]:
    print(f"{name:26s} R² (CV) = {cross_val_score(RandomForestRegressor(n_estimators=200, random_state=0), Xuse, yf, cv=cv, scoring='r2').mean():.4f}")
for name, Xuse in [("KNN, 25 feature", Xf), (f"KNN, {len(keep_idx)} feature (shadow)", Xf[:, keep_idx])]:
    knn = make_pipeline(StandardScaler(), KNeighborsRegressor(10))
    print(f"{name:26s} R² (CV) = {cross_val_score(knn, Xuse, yf, cv=cv, scoring='r2').mean():.4f}")

RF (có cơ chế tự chọn feature ở mỗi split) hưởng lợi **vừa phải** từ việc bỏ 20 feature nhiễu; KNN (dựa trên khoảng cách — mọi feature đóng góp như nhau) hưởng lợi **rất lớn**.

> ⚠️ Ô trên chọn feature bằng shadow trên **toàn bộ** dữ liệu rồi mới CV — chỉ để minh hoạ nhanh; ước lượng trung thực cần đặt bước chọn vào Pipeline/nested CV.

## 5. Tổng kết các phương pháp

| Họ | Ví dụ | Ưu | Nhược |
|---|---|---|---|
| Filter | variance, F-test, MI, chi² | rất nhanh, độc lập model | đánh giá từng feature: bỏ sót tương tác, không thấy dư thừa |
| Wrapper | forward/backward, RFE(CV) | thấy tương tác & dư thừa, theo đúng model | đắt, dễ overfit quá trình chọn |
| Embedded | Lasso, Elastic Net, SelectFromModel, shadow | chọn trong lúc train, cân bằng chi phí/chất lượng | phụ thuộc model; importance có thể lệch |

## 🧠 Kiểm tra nhanh

1. Vì sao L1 tạo hệ số đúng bằng 0 còn L2 thì không? (Gợi ý: hình học của ràng buộc — chương 06)
2. Stability selection giải quyết vấn đề gì?
3. Shadow feature là gì? Vì sao so với shadow **tốt nhất**?
4. Nêu 2 tình huống không nên chọn feature.

## 📌 Tóm tắt

- Embedded: L1/Elastic Net (hệ số = 0), importance + `SelectFromModel`, shadow features để có ngưỡng nhiễu khách quan.
- Feature tương quan làm lựa chọn không ổn định — Elastic Net, stability selection giúp.
- Chọn ≠ nhân quả; nhiều model hiện đại không cần chọn; luôn chọn trong CV.

## 📚 Tài liệu tham khảo

- Tibshirani (1996) — *Regression Shrinkage and Selection via the Lasso*
- Meinshausen & Bühlmann (2010) — *Stability selection*
- Kursa & Rudnicki (2010) — *Feature Selection with the Boruta Package*